In [5]:
final_features = [
    'launch_speed', 'launch_angle',
    'release_speed', 'release_spin_rate',
    'stand_enc', 'p_throws_enc',
    'pitcher_days_since_prev_game',
    'inning', 'balls', 'strikes',
    'park_hr_factor',
] + [col for col in fb.columns if col.startswith('pitch_') and fb[col].dtype != 'object']

X = fb[final_features].astype(float)
y = fb['is_hr']
dates = fb['game_date']

train_mask = dates.dt.year < 2025
test_mask = dates.dt.year == 2025

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f"Feature matrix shape: {X.shape}")
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print("Data loaded and ready")

Feature matrix shape: (130729, 28)
Train: 97,482 | Test: 33,247
Data loaded and ready


In [6]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    early_stopping_rounds=20,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

print("Training complete")

[0]	validation_0-logloss:0.41725
[50]	validation_0-logloss:0.20877
[100]	validation_0-logloss:0.19630
[150]	validation_0-logloss:0.19477
[200]	validation_0-logloss:0.19455
[205]	validation_0-logloss:0.19453
Training complete


In [7]:
from sklearn.metrics import roc_auc_score, log_loss, classification_report
import numpy as np

y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

auc = roc_auc_score(y_test, y_pred_proba)
ll = log_loss(y_test, y_pred_proba)

print(f"AUC:      {auc:.4f}")
print(f"Log Loss: {ll:.4f}")
print()
print(classification_report(y_test, y_pred))

AUC:      0.9530
Log Loss: 0.1945

              precision    recall  f1-score   support

           0       0.94      0.95      0.95     28075
           1       0.72      0.69      0.70      5172

    accuracy                           0.91     33247
   macro avg       0.83      0.82      0.83     33247
weighted avg       0.91      0.91      0.91     33247



In [8]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=final_features)
print(importance.sort_values(ascending=False).head(15))

launch_speed      0.657479
launch_angle      0.122202
park_hr_factor    0.018630
pitch_SI          0.015653
release_speed     0.015513
strikes           0.013515
pitch_KC          0.012679
pitch_ST          0.010914
balls             0.010545
pitch_FF          0.010176
pitch_FA          0.010151
stand_enc         0.009956
p_throws_enc      0.009479
pitch_SL          0.009162
pitch_number      0.008952
dtype: float32


In [9]:
import pickle
import os

model_path = os.path.join(os.path.expanduser("~"), "hr-projector", "models", "xgb_hr_model.pkl")

with open(model_path, 'wb') as f:
    pickle.dump(model, f)


features_path = os.path.join(os.path.expanduser("~"), "hr-projector", "models", "feature_cols.pkl")
with open(features_path, 'wb') as f:
    pickle.dump(final_features, f)

print(f"Model saved to {model_path}")
print(f"Features saved to {features_path}")

Model saved to /Users/arjangunsi/hr-projector/models/xgb_hr_model.pkl
Features saved to /Users/arjangunsi/hr-projector/models/feature_cols.pkl
